In [22]:
import os
from langchain_community.embeddings import HuggingFaceEmbeddings  # 本地模型，需先 pip install sentence-transformers
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",  # 本地服务可不校验，任意非空即可
    temperature=0,
    model="local",        # 若 LM Studio 要求指定模型名，可改为你在 LM Studio 里加载的模型名
)
# 相对路径：无论从项目根还是 notebookes 目录运行都能找到 test.csv
_csv = os.path.join("notebookes", "test.csv") if os.path.exists(os.path.join("notebookes", "test.csv")) else "test.csv"
loader = TextLoader(_csv)
documents = loader.load()

In [23]:
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
documents = text_splitter.split_documents(documents)
# 本地 embedding：首次运行会下载模型，之后离线可用。中文可改用 "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents, embeddings)
memory= ConversationBufferMemory(memory_key = "chat_history", return_messages = True)
qa = ConversationalRetrievalChain.from_llm(llm, vectorstore.as_retriever (), memory = memory)

In [24]:
query= "我该怎么阅读transformer模型？"
result= qa({"question": query})
print(result["answer"])

你可以按照以下步骤来阅读 Transformer 模型：

1. **先跑通**：从头执行到底，确认 loss 会降。确保模型能够正常训练并收敛。

2. **抓数据线**：理解数据处理流程，从句子开始，经过 `make_data`（补 padding）生成 `enc_inputs`、`dec_inputs`、`dec_outputs`（LongTensor），然后通过 `DataLoader` 批量化，再输入模型。

3. **看结构**：对照「代码结构图」，在 notebook 中找到 Transformer → Encoder → EncoderLayer → MultiHeadAttention → ScaledDotProductAttention 的结构，理解各模块之间的调用关系。

4. **看形状**：在关键处打印张量形状（如 `print(x.shape)`），对照「张量形状简图」确认维度变化，例如 `(B, L, d_model)` 等。

5. **抠细节**：深入理解核心组件，如 ScaledDotProductAttention 的 QK^T、mask、softmax、乘 V；Decoder 中的 dec_self_attn 与 dec_enc_attn 各自的计算过程。

**两种图各有用**：
- **代码结构图**：理清「谁调谁」。
- **矩阵/张量图**：理清「维度怎么变」。

按需画一种或两种图，不必一次画全。


In [26]:
query= "小桃是什么样的人？她讨厌裴青衫吗？"
result= qa({"question": query})
print(result["answer"])

小桃不是讨厌裴青衫的人。从文本中可以看出，她对裴青衫的态度是复杂而克制的：既有明确的规则和条件（比如约法三章、要求他还债才能外出花钱），也有对他的关心和照顾（如为他付医药费、留字条、陪他一起处理伤口、帮他买药和汤食等）。她对裴青衫的“不讨厌”体现在：

1. **不阻止他**：当裴青衫说“我也学学剑法怎么样？”时，小桃没有阻止，说明她并不排斥他。
2. **照顾他**：在裴青衫受伤后，她被救、被照顾、被送医、被送药、被送汤食，小桃都亲力亲为。
3. **不排斥他**：即使裴青衫偷了祭品、被小姑娘追打、被她揍，她也没有真正“讨厌”他，反而在事后还为他付医药费、留字条、陪他一起处理伤口。
4. **不排斥他**：即使裴青衫“偷了她的钱”，她也没有真正“讨厌”他，反而在事后还为他付医药费、留字条、陪他一起处理伤口。
5. **不排斥他**：即使裴青衫“偷了她的钱”，她也没有真正“讨厌”他，反而在事后还为他付医药费、留字条、陪他一起处理伤口。

因此，小桃并不讨厌裴青衫，而是对他保持一种“不排斥、不讨厌、但有规则”的态度。她对裴青衫的“不讨厌”体现在：她不阻止他、不排斥他、不讨厌他，而是对他保持一种“不排斥、不讨厌、但有规则”的态度。
